In [6]:
!pip install -q langchain langchain-community langchain-huggingface langchain_groq

In [19]:
from pydantic import BaseModel, Field, ConfigDict
from typing import Optional, List, Literal, Union
import os
import json
from google.colab import userdata
from enum import Enum
from datetime import datetime
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

In [15]:
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [20]:
# ___________Lesson Plan Parameters___________
class StudentLevel(str, Enum):
    beginner = "Beginner"
    intermediate = "Intermediate"
    advanced = "Advanced"


class LessonPlanRequest(BaseModel):
    model_config = ConfigDict(use_enum_values=True)

    subject: str = Field(..., description="Subject of the lesson")
    grade_level: str = Field(..., description="Grade level")
    topic: str = Field(..., description="Specific topic of the lesson")
    duration_minutes: int = Field(..., gt=0, le=240, description="Lesson duration in minutes")
    student_level: StudentLevel = Field(
        default=StudentLevel.intermediate,
        description="Overall proficiency level of the students"
    )
    learning_objectives: Optional[List[str]] = Field(
        default=None,
        description="Optional list of specific learning objectives / outcomes for the lesson"
    )

class LessonPlanOutput(BaseModel):
    title: str = Field(..., description="Title of the lesson")
    objectives: List[str] = Field(..., description="Learning objectives covered")
    materials: List[str] = Field(..., description="Materials/resources needed")
    warm_up: str = Field(..., description="Warm-up / hook activity")
    main_activities: List[str] = Field(..., description="Step-by-step main lesson activities")
    assessment: str = Field(..., description="How student understanding will be checked")
    closure: str = Field(..., description="Wrap-up / closing activity")


# ___________Quiz Generator Parameters___________
class Difficulty(str, Enum):
    easy = "Easy"
    medium = "Medium"
    hard = "Hard"
    mixed = "Mixed"


class QuestionType(str, Enum):
    multiple_choice = "Multiple Choice"
    true_false = "True/False"
    short_answer = "Short Answer"
    fill_in_the_blank = "Fill in the Blank"
    mixed = "Mixed"


class QuizGeneratorRequest(BaseModel):
    model_config = ConfigDict(use_enum_values=True)

    subject: str = Field(..., description="Subject of the quiz")
    grade_level: str = Field(..., description="Grade level, e.g. 'Grade 7'")
    topic: str = Field(..., description="Specific topic of the quiz")
    difficulty: Difficulty = Field(
        default=Difficulty.mixed,
        description="Difficulty level of the quiz questions"
    )
    number_of_questions: int = Field(..., gt=0, le=50, description="Total number of quiz questions to generate")
    question_type: QuestionType = Field(
        default=QuestionType.multiple_choice,
        description="Type of questions to generate"
    )

class QuizQuestion(BaseModel):
    question: str = Field(..., description="The question text")
    options: Optional[List[str]] = Field(default=None, description="Answer choices, if applicable")
    correct_answer: str = Field(..., description="The correct answer")
    explanation: Optional[str] = Field(default=None, description="Brief explanation of the correct answer")


class QuizOutput(BaseModel):
    title: str = Field(..., description="Title of the quiz")
    questions: List[QuizQuestion] = Field(..., description="List of generated quiz questions")


# ___________Worksheet Parameters___________
class WorksheetGeneratorRequest(BaseModel):
    model_config = ConfigDict(use_enum_values=True)

    subject: str = Field(..., description="Subject of the worksheet, e.g. 'Science'")
    grade_level: str = Field(..., description="Grade level, e.g. 'Grade 7'")
    difficulty: Difficulty = Field(
        default=Difficulty.medium,
        description="Difficulty level of the worksheet questions"
    )
    number_of_questions: int = Field(..., gt=0, le=50, description="Total number of worksheet questions to generate")
    topic: str = Field(..., description="Specific topic of the worksheet, e.g. 'Photosynthesis'")

class WorksheetProblem(BaseModel):
    prompt: str = Field(..., description="The worksheet question/problem text")
    answer: Optional[str] = Field(default=None, description="Answer key entry for this problem")


class WorksheetOutput(BaseModel):
    title: str = Field(..., description="Title of the worksheet")
    instructions: str = Field(..., description="Instructions shown to students at the top of the worksheet")
    problems: List[WorksheetProblem] = Field(..., description="List of worksheet problems with answer key")


# ___________Activity Parameters___________
class ActivityType(str, Enum):
    group_activity = "Group Activity"
    discussion = "Discussion"
    game = "Game"
    problem_solving = "Problem Solving"
    creative_activity = "Creative Activity"


class ActivityGeneratorRequest(BaseModel):
    model_config = ConfigDict(use_enum_values=True)

    subject: str = Field(..., description="Subject of the activity, e.g. 'Science'")
    grade_level: str = Field(..., description="Grade level, e.g. 'Grade 7'")
    duration_minutes: int = Field(..., gt=0, le=240, description="Activity duration in minutes")
    topic: str = Field(..., description="Specific topic of the activity, e.g. 'Photosynthesis'")
    activity_type: ActivityType = Field(
        default=ActivityType.group_activity,
        description="Type/format of the classroom activity to generate"
    )

class ActivityOutput(BaseModel):
    title: str = Field(..., description="Title of the activity")
    overview: str = Field(..., description="Brief overview of the activity")
    materials: List[str] = Field(..., description="Materials/resources needed")
    instructions: List[str] = Field(..., description="Step-by-step instructions for running the activity")
    wrap_up: str = Field(..., description="How to wrap up / debrief the activity")


# ___________AI Assistant Parameters___________
class QuickAction(str, Enum):
    create_lesson_plan = "Create a lesson plan"
    explain_topic = "Explain this topic simply"
    create_activity = "Create a classroom activity"
    generate_quiz_questions = "Generate quiz questions"
    analyze_student_performance = "Analyze student performance"
    write_learning_objectives = "Write learning objectives"


class ChatMessage(BaseModel):
    role: Literal["user", "assistant"] = Field(..., description="Who sent the message")
    content: str = Field(..., description="Message text")
    timestamp: Optional[datetime] = Field(default=None, description="When the message was sent")


class Attachment(BaseModel):
    filename: str = Field(..., description="Name of the attached file")
    file_type: Optional[str] = Field(default=None, description="MIME type or extension")
    url: Optional[str] = Field(default=None, description="Location/URL of the uploaded file")


class AIAssistantRequest(BaseModel):
    model_config = ConfigDict(use_enum_values=True)

    conversation_id: Optional[str] = Field(
        default=None,
        description="ID of an existing conversation to continue (e.g. 'Photosynthesis Lesson'); omit to start a new one"
    )
    message: str = Field(..., description="The user's current message/prompt, e.g. 'Ask anything about teaching...'")
    quick_action: Optional[QuickAction] = Field(
        default=None,
        description="Optional shortcut selected from the suggestion cards instead of typing a free-form message"
    )
    history: Optional[List[ChatMessage]] = Field(
        default=None,
        description="Prior turns in this conversation, for context"
    )
    attachments: Optional[List[Attachment]] = Field(
        default=None,
        description="Optional files attached via the paperclip icon"
    )

class ToolChoice(BaseModel):
    tool: Literal["lesson_plan", "quiz", "worksheet", "activity", "general"] = Field(
        ..., description="Which tool best answers the user's message"
    )


# ___________Teaching Material Parameters___________
class TeachingMaterialsRequest(BaseModel):
    model_config = ConfigDict(use_enum_values=True)

    subject: str = Field(..., description="Subject, e.g. 'Science'")
    grade_level: str = Field(..., description="Grade level, e.g. 'Grade 8'")
    topic: str = Field(..., description="Specific topic, e.g. 'Photosynthesis'")
    formats: List[Literal[
        "presentation_outline", "worksheet", "flashcards",
        "real_life_examples", "mcqs", "short_answer_questions", "discussion_activities"
    ]] = Field(
        default=[
            "presentation_outline", "worksheet", "flashcards",
            "real_life_examples", "mcqs", "short_answer_questions", "discussion_activities"
        ],
        description="Which material formats to generate from the same topic"
    )

class Flashcard(BaseModel):
    front: str = Field(..., description="Term or question side")
    back: str = Field(..., description="Definition or answer side")


class MaterialMCQ(BaseModel):
    question: str
    options: List[str]
    correct_answer: str


class TeachingMaterialsOutput(BaseModel):
    presentation_outline: Optional[List[str]] = Field(default=None, description="Slide-by-slide outline")
    worksheet: Optional[List[str]] = Field(default=None, description="Worksheet problems")
    flashcards: Optional[List[Flashcard]] = Field(default=None, description="Flashcard front/back pairs")
    real_life_examples: Optional[List[str]] = Field(default=None, description="Real-world examples of the topic")
    mcqs: Optional[List[MaterialMCQ]] = Field(default=None, description="Multiple choice questions")
    short_answer_questions: Optional[List[str]] = Field(default=None, description="Short-answer questions")
    discussion_activities: Optional[List[str]] = Field(default=None, description="Discussion prompts/activities")



# ___________Student Aware AI Parameters___________
class ConceptResult(BaseModel):
    concept: str = Field(..., description="Concept/topic name, e.g. 'Cellular Respiration'")
    students_struggling: int = Field(..., ge=0, description="Number of students struggling with this concept")
    total_students: Optional[int] = Field(default=None, description="Total students assessed, if known")


class StudentPerformanceRequest(BaseModel):
    model_config = ConfigDict(use_enum_values=True)

    subject: str = Field(..., description="Subject, e.g. 'Science'")
    grade_level: str = Field(..., description="Grade level, e.g. 'Grade 8'")
    assessment_name: Optional[str] = Field(default=None, description="Name of the quiz/assignment analyzed")
    concept_results: List[ConceptResult] = Field(..., description="Per-concept struggle counts from quiz/assignment results")


class ConceptInsight(BaseModel):
    concept: str
    students_struggling: int
    severity: Literal["low", "moderate", "high"] = Field(..., description="Urgency of intervention needed")
    recommendation: str = Field(..., description="Actionable recommendation, e.g. 'Consider revisiting this concept before moving forward.'")


class MiniLesson(BaseModel):
    concept: str
    title: str
    objective: str
    quick_explanation: str
    activity: str


class StudentPerformanceOutput(BaseModel):
    summary: str = Field(..., description="Overall plain-language summary of class performance")
    insights: List[ConceptInsight] = Field(..., description="Per-concept insights, ranked by severity")
    targeted_mini_lessons: List[MiniLesson] = Field(..., description="Mini re-teach lesson for each high-severity concept")



#___________Student Questions → Teacher Insights___________
class StudentQuestionsRequest(BaseModel):
    model_config = ConfigDict(use_enum_values=True)

    subject: str = Field(..., description="Subject, e.g. 'Science'")
    grade_level: str = Field(..., description="Grade level, e.g. 'Grade 8'")
    topic: str = Field(..., description="Topic the questions relate to, e.g. 'Photosynthesis'")
    student_questions: List[str] = Field(..., description="Raw list of questions submitted by students after the lesson/video")


class ConfusionTheme(BaseModel):
    theme: str = Field(..., description="Short label for the confusion cluster, e.g. 'Role of chlorophyll'")
    question_count: int = Field(..., ge=0, description="Number of student questions falling into this theme")
    example_questions: List[str] = Field(..., description="A few representative questions from this cluster")


class StudentQuestionsOutput(BaseModel):
    summary: str = Field(..., description="One-line summary, e.g. 'Students are mostly confused about...'")
    themes: List[ConfusionTheme] = Field(..., description="Confusion themes ranked by question_count descending")
    clarification_lesson: MiniLesson = Field(..., description="A clarification mini-lesson addressing the top theme(s)")


In [17]:
llm_primary = ChatGroq(model='openai/gpt-oss-20b', temperature=0, max_tokens=8192)
llm_fallback = ChatGroq(model='openai/gpt-oss-120b', temperature=0, max_tokens=8192)

class LLMRouter:
    """
    Thin routing wrapper around two LangChain chat models.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        primary: LangChain chat model, tried first on every invoke().
        fallback: LangChain chat model, used only if primary raises.
      behavior:
        invoke(prompt_str) calls primary.invoke(...); on any exception it
        logs the failure and retries once against fallback.invoke(...).
      status: active
    """
    def __init__(self, primary, fallback):
        self.primary = primary
        self.fallback = fallback

    def invoke(self, prompt_str: str):
        try:
            return self.primary.invoke(prompt_str)
        except Exception as e:
            print(f"[LLMRouter] Primary model failed ({e}), falling back...")
            return self.fallback.invoke(prompt_str)

llm = LLMRouter(llm_primary, llm_fallback)

In [ ]:
lesson_plan_parser = PydanticOutputParser(pydantic_object=LessonPlanOutput)

lesson_plan_prompt = PromptTemplate(
    template=(
        "You are an expert curriculum designer. Create a detailed lesson plan.\n"
        "Subject: {subject}\n"
        "Grade Level: {grade_level}\n"
        "Topic: {topic}\n"
        "Duration: {duration_minutes} minutes\n"
        "Student Level: {student_level}\n"
        "Learning Objectives (if provided): {learning_objectives}\n\n"
        "{format_instructions}"
    ),
    input_variables=["subject", "grade_level", "topic", "duration_minutes", "student_level", "learning_objectives"],
    partial_variables={"format_instructions": lesson_plan_parser.get_format_instructions()}
)


def generate_lesson_plan(request: LessonPlanRequest) -> dict:
    chain = lesson_plan_prompt | llm.primary | lesson_plan_parser
    try:
        result = chain.invoke({
            "subject": request.subject,
            "grade_level": request.grade_level,
            "topic": request.topic,
            "duration_minutes": request.duration_minutes,
            "student_level": request.student_level,
            "learning_objectives": request.learning_objectives or "None specified"
        })
    except Exception as e:
        print(f"[LessonPlanner] Primary failed ({e}), falling back...")
        chain = lesson_plan_prompt | llm.fallback | lesson_plan_parser
        result = chain.invoke({
            "subject": request.subject,
            "grade_level": request.grade_level,
            "topic": request.topic,
            "duration_minutes": request.duration_minutes,
            "student_level": request.student_level,
            "learning_objectives": request.learning_objectives or "None specified"
        })
    return json.loads(result.model_dump_json())

In [ ]:
quiz_parser = PydanticOutputParser(pydantic_object=QuizOutput)

quiz_prompt = PromptTemplate(
    template=(
        "You are an expert teacher creating a quiz.\n"
        "Subject: {subject}\n"
        "Grade Level: {grade_level}\n"
        "Topic: {topic}\n"
        "Difficulty: {difficulty}\n"
        "Number of Questions: {number_of_questions}\n"
        "Question Type: {question_type}\n\n"
        "{format_instructions}"
    ),
    input_variables=["subject", "grade_level", "topic", "difficulty", "number_of_questions", "question_type"],
    partial_variables={"format_instructions": quiz_parser.get_format_instructions()}
)


def generate_quiz(request: QuizGeneratorRequest) -> dict:
    chain = quiz_prompt | llm.primary | quiz_parser
    inputs = {
        "subject": request.subject,
        "grade_level": request.grade_level,
        "topic": request.topic,
        "difficulty": request.difficulty,
        "number_of_questions": request.number_of_questions,
        "question_type": request.question_type
    }
    try:
        result = chain.invoke(inputs)
    except Exception as e:
        print(f"[QuizGenerator] Primary failed ({e}), falling back...")
        chain = quiz_prompt | llm.fallback | quiz_parser
        result = chain.invoke(inputs)
    return json.loads(result.model_dump_json())

In [ ]:
worksheet_parser = PydanticOutputParser(pydantic_object=WorksheetOutput)

worksheet_prompt = PromptTemplate(
    template=(
        "You are an expert teacher creating a print-ready worksheet.\n"
        "Subject: {subject}\n"
        "Grade Level: {grade_level}\n"
        "Topic: {topic}\n"
        "Difficulty: {difficulty}\n"
        "Number of Questions: {number_of_questions}\n\n"
        "{format_instructions}"
    ),
    input_variables=["subject", "grade_level", "topic", "difficulty", "number_of_questions"],
    partial_variables={"format_instructions": worksheet_parser.get_format_instructions()}
)


def generate_worksheet(request: WorksheetGeneratorRequest) -> dict:
    chain = worksheet_prompt | llm.primary | worksheet_parser
    inputs = {
        "subject": request.subject,
        "grade_level": request.grade_level,
        "topic": request.topic,
        "difficulty": request.difficulty,
        "number_of_questions": request.number_of_questions
    }
    try:
        result = chain.invoke(inputs)
    except Exception as e:
        print(f"[WorksheetGenerator] Primary failed ({e}), falling back...")
        chain = worksheet_prompt | llm.fallback | worksheet_parser
        result = chain.invoke(inputs)
    return json.loads(result.model_dump_json())

In [ ]:
activity_parser = PydanticOutputParser(pydantic_object=ActivityOutput)

activity_prompt = PromptTemplate(
    template=(
        "You are an expert teacher designing an engaging classroom activity.\n"
        "Subject: {subject}\n"
        "Grade Level: {grade_level}\n"
        "Topic: {topic}\n"
        "Duration: {duration_minutes} minutes\n"
        "Activity Type: {activity_type}\n\n"
        "{format_instructions}"
    ),
    input_variables=["subject", "grade_level", "topic", "duration_minutes", "activity_type"],
    partial_variables={"format_instructions": activity_parser.get_format_instructions()}
)


def generate_activity(request: ActivityGeneratorRequest) -> dict:
    chain = activity_prompt | llm.primary | activity_parser
    inputs = {
        "subject": request.subject,
        "grade_level": request.grade_level,
        "topic": request.topic,
        "duration_minutes": request.duration_minutes,
        "activity_type": request.activity_type
    }
    try:
        result = chain.invoke(inputs)
    except Exception as e:
        print(f"[ActivityGenerator] Primary failed ({e}), falling back...")
        chain = activity_prompt | llm.fallback | activity_parser
        result = chain.invoke(inputs)
    return json.loads(result.model_dump_json())

In [ ]:
tool_choice_parser = PydanticOutputParser(pydantic_object=ToolChoice)

router_prompt = PromptTemplate(
    template=(
        "Decide which teaching tool should handle this user request.\n"
        "- lesson_plan: user wants a full lesson plan\n"
        "- quiz: user wants quiz/test questions\n"
        "- worksheet: user wants a printable worksheet\n"
        "- activity: user wants a classroom activity/game/discussion\n"
        "- general: anything else (explaining a concept, analysis, general Q&A)\n\n"
        "User message: {message}\n\n"
        "{format_instructions}"
    ),
    input_variables=["message"],
    partial_variables={"format_instructions": tool_choice_parser.get_format_instructions()}
)

extraction_parsers = {
    "lesson_plan": PydanticOutputParser(pydantic_object=LessonPlanRequest),
    "quiz": PydanticOutputParser(pydantic_object=QuizGeneratorRequest),
    "worksheet": PydanticOutputParser(pydantic_object=WorksheetGeneratorRequest),
    "activity": PydanticOutputParser(pydantic_object=ActivityGeneratorRequest),
}

extraction_prompt_template = PromptTemplate(
    template=(
        "Extract structured parameters from the user's message to call the {tool_name} tool. "
        "If a field isn't mentioned, make a reasonable assumption appropriate for a classroom teacher.\n\n"
        "User message: {message}\n\n"
        "{format_instructions}"
    ),
    input_variables=["tool_name", "message"],
    partial_variables={}
)

tool_dispatch = {
    "lesson_plan": generate_lesson_plan,
    "quiz": generate_quiz,
    "worksheet": generate_worksheet,
    "activity": generate_activity,
}


def _invoke_with_fallback(chain_primary, chain_fallback, inputs):
    try:
        return chain_primary.invoke(inputs)
    except Exception as e:
        print(f"[Orchestrator] Primary failed ({e}), falling back...")
        return chain_fallback.invoke(inputs)


def ai_assistant(request: AIAssistantRequest) -> dict:
    message = request.quick_action if request.quick_action else request.message

    router_chain_p = router_prompt | llm.primary | tool_choice_parser
    router_chain_f = router_prompt | llm.fallback | tool_choice_parser
    tool_choice = _invoke_with_fallback(router_chain_p, router_chain_f, {"message": message})

    if tool_choice.tool == "general":
        general_prompt = PromptTemplate(
            template="You are a helpful AI teaching copilot. Answer the teacher's question clearly and practically.\n\nQuestion: {message}",
            input_variables=["message"]
        )
        chain_p = general_prompt | llm.primary
        chain_f = general_prompt | llm.fallback
        response = _invoke_with_fallback(chain_p, chain_f, {"message": message})
        return {
            "tool_used": "general",
            "response": response.content if hasattr(response, "content") else str(response)
        }

    parser = extraction_parsers[tool_choice.tool]
    extraction_prompt = PromptTemplate(
        template=extraction_prompt_template.template,
        input_variables=["tool_name", "message"],
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )
    chain_p = extraction_prompt | llm.primary | parser
    chain_f = extraction_prompt | llm.fallback | parser
    extracted_request = _invoke_with_fallback(
        chain_p, chain_f, {"tool_name": tool_choice.tool, "message": message}
    )

    result = tool_dispatch[tool_choice.tool](extracted_request)

    return {
        "tool_used": tool_choice.tool,
        "parameters": json.loads(extracted_request.model_dump_json()),
        "result": result
    }

In [ ]:
materials_parser = PydanticOutputParser(pydantic_object=TeachingMaterialsOutput)

materials_prompt = PromptTemplate(
    template=(
        "You are an expert teacher creating multiple teaching materials from a single topic, "
        "so the teacher doesn't have to rewrite the same content repeatedly.\n"
        "Subject: {subject}\n"
        "Grade Level: {grade_level}\n"
        "Topic: {topic}\n"
        "Only generate these formats, leave the rest null: {formats}\n\n"
        "{format_instructions}"
    ),
    input_variables=["subject", "grade_level", "topic", "formats"],
    partial_variables={"format_instructions": materials_parser.get_format_instructions()}
)


def generate_teaching_materials(request: TeachingMaterialsRequest) -> dict:
    chain = materials_prompt | llm.primary | materials_parser
    inputs = {
        "subject": request.subject,
        "grade_level": request.grade_level,
        "topic": request.topic,
        "formats": ", ".join(request.formats)
    }
    try:
        result = chain.invoke(inputs)
    except Exception as e:
        print(f"[TeachingMaterials] Primary failed ({e}), falling back...")
        chain = materials_prompt | llm.fallback | materials_parser
        result = chain.invoke(inputs)
    return json.loads(result.model_dump_json())

In [ ]:
performance_parser = PydanticOutputParser(pydantic_object=StudentPerformanceOutput)

performance_prompt = PromptTemplate(
    template=(
        "You are an expert teaching analyst. A teacher uploaded quiz/assignment results broken down by concept.\n"
        "Subject: {subject}\n"
        "Grade Level: {grade_level}\n"
        "Assessment: {assessment_name}\n"
        "Concept results (concept, students_struggling, total_students):\n{concept_results}\n\n"
        "Identify which concepts need the most attention, explain severity, give a clear recommendation per concept "
        "(e.g. 'X students are struggling with Y. Consider revisiting this concept before moving forward.'), "
        "and generate a targeted mini-lesson for every concept marked 'high' severity.\n\n"
        "{format_instructions}"
    ),
    input_variables=["subject", "grade_level", "assessment_name", "concept_results"],
    partial_variables={"format_instructions": performance_parser.get_format_instructions()}
)


def analyze_student_performance(request: StudentPerformanceRequest) -> dict:
    concept_lines = "\n".join(
        f"- {c.concept}: {c.students_struggling} struggling"
        + (f" / {c.total_students} total" if c.total_students else "")
        for c in request.concept_results
    )
    inputs = {
        "subject": request.subject,
        "grade_level": request.grade_level,
        "assessment_name": request.assessment_name or "Unnamed assessment",
        "concept_results": concept_lines
    }
    chain = performance_prompt | llm.primary | performance_parser
    try:
        result = chain.invoke(inputs)
    except Exception as e:
        print(f"[StudentPerformance] Primary failed ({e}), falling back...")
        chain = performance_prompt | llm.fallback | performance_parser
        result = chain.invoke(inputs)
    return json.loads(result.model_dump_json())

In [ ]:
questions_parser = PydanticOutputParser(pydantic_object=StudentQuestionsOutput)

questions_prompt = PromptTemplate(
    template=(
        "You are an expert teaching analyst. Students submitted questions after a lesson. "
        "Cluster the questions into confusion themes (do not use quiz scores, use the questions themselves).\n"
        "Subject: {subject}\n"
        "Grade Level: {grade_level}\n"
        "Topic: {topic}\n"
        "Student questions:\n{student_questions}\n\n"
        "Rank themes by how many questions fall into each, and generate one clarification mini-lesson "
        "targeting the biggest theme(s).\n\n"
        "{format_instructions}"
    ),
    input_variables=["subject", "grade_level", "topic", "student_questions"],
    partial_variables={"format_instructions": questions_parser.get_format_instructions()}
)


def analyze_student_questions(request: StudentQuestionsRequest) -> dict:
    inputs = {
        "subject": request.subject,
        "grade_level": request.grade_level,
        "topic": request.topic,
        "student_questions": "\n".join(f"- {q}" for q in request.student_questions)
    }
    chain = questions_prompt | llm.primary | questions_parser
    try:
        result = chain.invoke(inputs)
    except Exception as e:
        print(f"[StudentQuestions] Primary failed ({e}), falling back...")
        chain = questions_prompt | llm.fallback | questions_parser
        result = chain.invoke(inputs)
    return json.loads(result.model_dump_json())